# H14 · Convergence: are country gaps in company profitability closing?

> **Hypothesis H14:** The gap in company ROA between less and more developed countries narrows over time, so the (negative) development–ROA slope moves towards zero.

| | |
|---|---|
| **Why we ask** | H0 says firms in poorer countries earn more. Classical growth theory says gaps close as poorer economies catch up: higher returns attract capital, which pushes returns down. If that is happening, the slope should flatten year by year. We also ask the textbook version for whole economies (β-convergence of GDP per person). |
| **Prediction** | **Positive** interaction between the development index and time (the negative slope flattens). |
| **Data** | Yahoo company-years, fiscal years 2022–2025 (t = fiscal year − 2022; 2021 has only 89 company-years because Yahoo provides about four years of statements). Development index of the previous year. |
| **Primary test (fixed before looking at the result)** | `ROA (pp) ~ development index × t + sector + size rank + fiscal year`; the test is the coefficient on development index × t (change of the development slope per year). Errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra="", controls=CONTROLS, data=None):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy if data is None else data), outcome=outcome, extra=extra, B=B, controls=controls)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · The slope year by year
The development–ROA coefficient estimated separately for each fiscal year, with 95% intervals. If the gap closes, the bars rise towards zero.

In [2]:
f5 = fy[fy.fiscal_year.between(2022, 2025)].copy()
f5["t"] = f5.fiscal_year - 2022; f5["dev_x_t"] = f5.dev_z * f5.t
rows = []
for y, g in f5.groupby("fiscal_year"):
    m = fit_cluster("roa_pp ~ dev_z + C(sector) + size_rank", g.dropna(subset=["dev_z", "size_rank"]))
    rows.append({"fiscal year": y, "coef": m.params["dev_z"], "lo": m.params["dev_z"] - 1.96 * m.bse["dev_z"], "hi": m.params["dev_z"] + 1.96 * m.bse["dev_z"], "firm-years": int(m.nobs), "median ROA": g.roa_pp.median()})
by_year = pd.DataFrame(rows).set_index("fiscal year")
fig = go.Figure(go.Scatter(x=by_year.index, y=by_year.coef, mode="lines+markers", error_y=dict(type="data", array=by_year.hi - by_year.coef, arrayminus=by_year.coef - by_year.lo)))
fig.add_hline(y=0, line_color="grey"); fig.update_layout(height=380, title="Development → ROA (pp per +1 SD), by fiscal year", xaxis_title="fiscal year", yaxis_title="pp"); fig.show()
by_year.round(2)

,coef,lo,hi,firm-years,median ROA
fiscal year,,,,,
2022,-1.000,-1.380,-0.620,2528,3.840
2023,-1.130,-1.680,-0.580,2547,3.620
2024,-1.220,-1.760,-0.680,2547,3.780
2025,-1.220,-1.780,-0.670,2507,3.840


### Do the countries' ROAs get closer together? (σ-convergence)
The spread (standard deviation) of country median ROAs by year, countries with at least 10 firms in the year.

In [3]:
cy = f5.groupby(["fiscal_year", "iso3"]).agg(roa=("roa_pp", "median"), n=("roa_pp", "size")).query("n >= @MIN_FIRMS").reset_index()
sig = cy.groupby("fiscal_year").agg(sd=("roa", "std"), iqr=("roa", lambda s: s.quantile(.75) - s.quantile(.25)), countries=("roa", "size"))
fig = px.line(sig.reset_index(), x="fiscal_year", y=["sd", "iqr"], markers=True, title="Spread of country median ROA across countries (pp)"); fig.update_layout(height=340); fig.show()
sig.round(2)

,sd,iqr,countries
fiscal_year,,,
2022,1.360,1.600,33
2023,1.380,1.540,33
2024,1.370,1.520,33
2025,1.290,1.650,33


## 2 · Primary test
The slope of the development effect over time, estimated in one model with the development × t interaction.

In [4]:
primary = wild_cluster("dev_x_t", f5, outcome="roa_pp", extra="dev_z", B=1999)
print(f"Change in the development effect per year: {primary['coef']:+.2f} pp per +1 SD per year (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}); over the three years 2022→2025 {3 * primary['coef']:+.2f} pp.")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f} | {primary['clusters']} countries, {primary['n']:,} company-years")

Change in the development effect per year: -0.05 pp per +1 SD per year (95% CI -0.19 to +0.08); over the three years 2022→2025 -0.16 pp.
p (cluster-robust) = 0.4312 | p (wild bootstrap) = 0.4715 | 34 countries, 10,129 company-years


## 3 · The textbook version: β-convergence of GDP per person
Do economies that started poorer in 2012 grow faster to 2024? Annual growth of GDP per person (PPP, log difference) against the starting level, 34 countries, plus an EU-only version.

In [5]:
g = panel.set_index(["iso3", "year"]).gdp_per_capita_ppp.unstack()          # already in logs
a, b = 2012, 2024
bc = pd.DataFrame({"start": g[a], "growth": (g[b] - g[a]) / (b - a) * 100}).dropna().join(dim[["name", "is_eu"]])
def beta_test(d):
    m = smf.ols("growth ~ start", d).fit(cov_type="HC3")
    r, p, n = spearman(d.start, d.growth)
    rng = np.random.default_rng(0)
    perm = np.array([stats.spearmanr(rng.permutation(d.start.values), d.growth.values)[0] for _ in range(5000)])
    return {"slope per log point": m.params["start"], "ci_low": m.conf_int().loc["start", 0], "ci_high": m.conf_int().loc["start", 1], "p (HC3)": m.pvalues["start"], "Spearman ρ": r, "permutation p": (np.sum(np.abs(perm) >= abs(r)) + 1) / 5001, "countries": n}
beta = pd.DataFrame({"all 34 countries": beta_test(bc), "EU only": beta_test(bc[bc.is_eu])}).T
fig = px.scatter(bc.reset_index(), x="start", y="growth", text="iso3", color="is_eu", hover_name="name", title=f"Growth of GDP per person {a}–{b} vs level in {a}", labels={"start": f"log GDP per person (PPP) {a}", "growth": "annual growth (%)"}, height=450)
fig.update_traces(textposition="top center"); fig.show()
beta

,slope per log point,ci_low,ci_high,p (HC3),Spearman ρ,permutation p,countries
all 34 countries,-1.013,-1.949,-0.076,0.034,-0.319,0.075,34.000
EU only,-1.914,-3.067,-0.761,0.001,-0.678,0.002,21.000


## 4 · Robustness
The interaction in seven samples (wild bootstrap, 499 draws each).

In [6]:
rob, rob_share = robustness("dev_x_t", "roa_pp", 1, extra="dev_z", data=f5)
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="Change of the development slope per year (pp) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Predicted sign (positive) in {rob_share:.0%} of samples.")
rob

Predicted sign (positive) in 0% of samples.


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,-0.054,-0.187,0.079,0.484,34,10129,False
excluding financials & real estate,-0.073,-0.238,0.092,0.420,34,7995,False
excluding small-sample countries,-0.052,-0.187,0.083,0.444,29,9698,False
"excluding USA, China, Japan",-0.049,-0.179,0.082,0.478,31,8935,False
fiscal years 2023 and later,-0.017,-0.198,0.165,0.878,34,7601,False
EU countries only,-0.075,-0.428,0.277,0.662,21,5069,False
non-EU countries only,-0.015,-0.161,0.131,0.840,13,5060,False


## 5 · Replication on ESEF
EU companies over a longer window (fiscal 2018–2025), so the time trend has more room.

In [7]:
fe = prep.load_firms(con, country, source="ESEF"); fe = fe[fe.fiscal_year.between(2018, 2025)].copy()
fe["t"] = fe.fiscal_year - 2018; fe["dev_x_t"] = fe.dev_z * fe.t
rep = wild_cluster("dev_x_t", fe, outcome="roa_pp", extra="dev_z", controls="size_rank + C(fiscal_year)", B=1999)
rep_same = bool(np.sign(rep["coef"]) == 1)
print(f"ESEF 2018–2025 ({rep['n']:,} company-years, {rep['clusters']} countries): {rep['coef']:+.2f} pp per year (95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")

ESEF 2018–2025 (5,312 company-years, 19 countries): +0.06 pp per year (95% CI -0.71 to +0.83), wild p = 0.893


## 6 · Verdict

In [8]:
v = utils.verdict(primary["coef"], primary["p_wild"], 1, rob_share, primary["clusters"])
ba = beta.loc["all 34 countries"]
lines = [
 f"Primary: the development slope changes by {primary['coef']:+.2f} pp per +1 SD per year (from {by_year.coef.iloc[0]:+.2f} in {by_year.index[0]} to {by_year.coef.iloc[-1]:+.2f} in {by_year.index[-1]}), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} company-years.",
 f"Robustness: predicted sign in {rob_share:.0%} of seven samples. ESEF 2018–2025: {rep['coef']:+.2f} pp per year (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"σ-convergence: SD of country median ROA {sig.sd.iloc[0]:.2f} in {sig.index[0]} vs {sig.sd.iloc[-1]:.2f} in {sig.index[-1]}.",
 f"β-convergence of GDP per person 2012–2024: slope {ba['slope per log point']:+.2f} (ρ = {ba['Spearman ρ']:+.2f}, permutation p = {ba['permutation p']:.3f}, n = {int(ba['countries'])}); EU only ρ = {beta.loc['EU only', 'Spearman ρ']:+.2f} (p = {beta.loc['EU only', 'permutation p']:.3f}).",
 f"VERDICT H14: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H14", id="H14", title="Convergence of company profitability", statement="The development–ROA gap narrows over time.",
                  expected_sign=1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp ROA per +1 SD development per year (development × time)",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.14_convergence.ipynb", notes=lines)

Primary: the development slope changes by -0.05 pp per +1 SD per year (from -1.00 in 2022 to -1.22 in 2025), wild p = 0.471, 34 countries, 10,129 company-years.
Robustness: predicted sign in 0% of seven samples. ESEF 2018–2025: +0.06 pp per year (wild p = 0.893), same direction.
σ-convergence: SD of country median ROA 1.36 in 2022 vs 1.29 in 2025.
β-convergence of GDP per person 2012–2024: slope -1.01 (ρ = -0.32, permutation p = 0.075, n = 34); EU only ρ = -0.68 (p = 0.002).
VERDICT H14: NOT SUPPORTED


### Limits
- Five fiscal years is short for convergence, and they include the pandemic recovery and the 2022–2023 inflation shock; any trend may be the cycle, not catch-up.
- The sample of listed companies changes slightly from year to year (new listings, missing filings), which can move group averages.
- β-convergence over 2012–2024 uses only 34 countries and includes the global financial crisis aftermath; the EU subset is dominated by Eastern Europe catching up.
- Association, not causation.

In [9]:
con.close()